In [14]:
import os, glob, subprocess
import multiprocessing as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
info = pd.read_csv("../20221221_qc/results/NASCseq_Summary.csv")
info = info[info["Stranded.Reads"] >= 200000]
info1 = info[[run.startswith("GSE") for run in info["Run"]]]
info2 = info[[not run.startswith("GSE") for run in info["Run"]]]

groups = ["GSE128273_NASCseq", "NASCseq"]

annfile = "../../../share/homo_sapiens/GRCh38.p13/gencode.v39.transcript_info.added_canonical.csv"

def run_cmd(cmd):
    subprocess.check_call(cmd, shell=True)

pool = mp.Pool(24)
print("s4U\tTime\tCells\tName")
for group, info in zip(groups, [info1, info2]):
    for s4u, tmp1 in info.groupby(by="s4U"):
        for time, tmp2 in tmp1.groupby(by="Time"):
            name = "%s.s4U_%duM_%s" % (group, s4u, "%dmin" % (time*60) if time < 1 else "%dh" % time)
            print(s4u, time, len(tmp2), name, sep="\t")
            
            path1 = f"results/pseudobulk/{name}.filelist.txt"
            path2 = f"results/pseudobulk/{name}.tsv.gz"
            ! mkdir -p `dirname {path1}`
            
            with open(path1, "w+") as fw:
                for cell in tmp2["Cell"]:
                    run = cell.split(".")[0]
                    path = f"../../results/1_nascseq/4_quantify/fpkm/marknew/{run}/{cell}.tsv"
                    fw.write(path + "\n")

            if not os.path.exists(path2):
                time2 = 0 if s4u == 0 else time
                cmd = f"python merge_counts.NASCseq.py {path1} {annfile} {time2} {path2}"
                pool.apply_async(run_cmd, (cmd, ))

pool.close()
pool.join()

s4U	Time	Cells	Name
0	3.0	14	GSE128273_NASCseq.s4U_0uM_3h
50	0.25	45	GSE128273_NASCseq.s4U_50uM_15min
50	1.0	122	GSE128273_NASCseq.s4U_50uM_1h
50	3.0	79	GSE128273_NASCseq.s4U_50uM_3h
0	3.0	39	NASCseq.s4U_0uM_3h
50	2.0	10	NASCseq.s4U_50uM_2h
50	3.0	38	NASCseq.s4U_50uM_3h
100	2.0	10	NASCseq.s4U_100uM_2h
100	3.0	12	NASCseq.s4U_100uM_3h
200	2.0	12	NASCseq.s4U_200uM_2h
200	3.0	9	NASCseq.s4U_200uM_3h
500	2.0	12	NASCseq.s4U_500uM_2h
